# Vyuha P12 - L3 MCP tool-poisoning defense

**MCP tool poisoning** hides instructions to the agent inside a tool's *description* or *parameter metadata* - read and trusted before the tool ever runs. `MCPToolScanner` screens each tool definition at discovery time (de-obfuscating first, then applying the L3 injection rules + MCP-specific patterns) and drops poisoned tools before they reach the agent's context.

Maps to **OWASP Top 10 for Agentic Applications 2026** (ASI02 Tool Misuse, ASI06 Memory/Context Poisoning). CPU-only - no GPU or API key needed.

In [ ]:
import sys, os, glob, subprocess
REPO_URL = "https://github.com/g25ait2149/vyuha.git"
DEST = "/kaggle/working/vyuha_src"
if os.path.isdir(os.path.join(DEST, ".git")):
    subprocess.run(["git", "-C", DEST, "pull", "--ff-only"], check=False)
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, DEST], check=False)
hits = glob.glob(DEST + "/**/vyuha/__init__.py", recursive=True)
root = os.path.dirname(os.path.dirname(hits[0])) if hits else DEST
sys.path.insert(0, root)
for m in [m for m in sys.modules if m == "vyuha" or m.startswith(("vyuha.", "eval"))]:
    del sys.modules[m]
print("vyuha repo at:", root)

## Run: MCP tool-poisoning detection
Detection rate on poisoned tools (hidden instructions, exfiltration, conditional triggers, an obfuscated one) vs pass-rate on legitimate tools that merely *describe* sending/reading/transferring.

In [ ]:
from eval.mcp_eval import mcp_poisoning_eval
results = mcp_poisoning_eval()
results

## Inspect per-tool verdicts

In [ ]:
from vyuha.agent import MCPToolScanner
from eval.mcp_eval import POISONED, BENIGN
sc = MCPToolScanner()
print('POISONED (should all be caught):')
for t in POISONED:
    v = sc.scan_tool(t); print(f"  {t['name']:<12} poisoned={v['poisoned']} score={v['score']} rules={v['rules']}")
print('\nBENIGN (should all pass):')
for t in BENIGN:
    v = sc.scan_tool(t); print(f"  {t['name']:<14} poisoned={v['poisoned']} score={v['score']}")